In [1]:
import os
print(os.getcwd())

E:\icu_project


In [2]:
import pandas as pd

icu_hourly = pd.read_csv("data/processed_icu_hourly_v1.csv")

print(icu_hourly.shape)
icu_hourly.head()

(149775, 16)


,patientunitstayid,hour,sao2,heartrate,respiration,BUN,HCO3,Hct,Hgb,WBC x 1000,creatinine,magnesium,pH,platelets x 1000,potassium,sodium
0,141764,0,NaN,117.750000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,141764,1,NaN,107.666667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,141764,2,NaN,95.666667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,141764,3,NaN,106.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,141764,4,NaN,107.833333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
patients = pd.read_csv("data/eicu_demo/patient.csv")

patients[["patientunitstayid", "hospitaldischargestatus"]].head()

,patientunitstayid,hospitaldischargestatus
0,141764,Alive
1,141765,Alive
2,143870,Alive
3,144815,Alive
4,145427,Alive


In [4]:
patients["hospitaldischargestatus"].value_counts()

hospitaldischargestatus
Alive      2280
Expired     212
Name: count, dtype: int64

In [5]:
patients["mortality"] = (
    patients["hospitaldischargestatus"]
    .map({"Alive": 0, "Expired": 1})
)

patients["mortality"].value_counts()

mortality
0.0    2280
1.0     212
Name: count, dtype: int64

In [6]:
icu_labeled = icu_hourly.merge(
    patients[["patientunitstayid", "mortality"]],
    on="patientunitstayid",
    how="left"
)

print("Total rows:", icu_labeled.shape[0])
print("Unique patients:", icu_labeled["patientunitstayid"].nunique())
print("Mortality distribution:")
print(icu_labeled["mortality"].value_counts())

Total rows: 149775
Unique patients: 2468
Mortality distribution:
mortality
0.0    129821
1.0     18271
Name: count, dtype: int64


In [7]:
import numpy as np

unique_patients = icu_labeled["patientunitstayid"].unique()

np.random.seed(42)
np.random.shuffle(unique_patients)

split_index = int(0.8 * len(unique_patients))

train_patients = unique_patients[:split_index]
test_patients = unique_patients[split_index:]

print("Train patients:", len(train_patients))
print("Test patients:", len(test_patients))

Train patients: 1974
Test patients: 494


In [8]:
train_df = icu_labeled[
    icu_labeled["patientunitstayid"].isin(train_patients)
]

test_df = icu_labeled[
    icu_labeled["patientunitstayid"].isin(test_patients)
]

print("Train rows:", train_df.shape[0])
print("Test rows:", test_df.shape[0])

Train rows: 117840
Test rows: 31935


In [9]:
X_train = train_df.drop(columns=["patientunitstayid", "mortality"])
y_train = train_df["mortality"]

X_test = test_df.drop(columns=["patientunitstayid", "mortality"])
y_test = test_df["mortality"]

print(X_train.shape)

(117840, 15)


In [10]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Imputer
imputer = SimpleImputer(strategy="median")

X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

# Scale
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

In [11]:
icu_labeled["mortality"].isna().sum()

np.int64(1683)

In [12]:
icu_labeled = icu_labeled.dropna(subset=["mortality"])

print("New shape:", icu_labeled.shape)
print("Mortality distribution:")
print(icu_labeled["mortality"].value_counts())

New shape: (148092, 17)
Mortality distribution:
mortality
0.0    129821
1.0     18271
Name: count, dtype: int64


In [13]:
unique_patients = icu_labeled["patientunitstayid"].unique()

np.random.seed(42)
np.random.shuffle(unique_patients)

split_index = int(0.8 * len(unique_patients))

train_patients = unique_patients[:split_index]
test_patients = unique_patients[split_index:]

train_df = icu_labeled[
    icu_labeled["patientunitstayid"].isin(train_patients)
]

test_df = icu_labeled[
    icu_labeled["patientunitstayid"].isin(test_patients)
]

X_train = train_df.drop(columns=["patientunitstayid", "mortality"])
y_train = train_df["mortality"]

X_test = test_df.drop(columns=["patientunitstayid", "mortality"])
y_test = test_df["mortality"]

print("Train rows:", X_train.shape)
print("Test rows:", X_test.shape)

Train rows: (117943, 15)
Test rows: (30149, 15)


In [14]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Impute
imputer = SimpleImputer(strategy="median")

X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

# Scale
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

In [15]:
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

logreg.fit(X_train_scaled, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [16]:
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

y_pred = logreg.predict(X_test_scaled)
y_prob = logreg.predict_proba(X_test_scaled)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print(classification_report(y_test, y_pred))

Accuracy: 0.7489468970778467
ROC-AUC: 0.7174855975007515
              precision    recall  f1-score   support

         0.0       0.93      0.78      0.85     26918
         1.0       0.22      0.52      0.31      3231

    accuracy                           0.75     30149
   macro avg       0.58      0.65      0.58     30149
weighted avg       0.86      0.75      0.79     30149



In [17]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_imputed, y_train)

y_pred_rf = rf.predict(X_test_imputed)
y_prob_rf = rf.predict_proba(X_test_imputed)[:, 1]

print("RF Accuracy:", accuracy_score(y_test, y_pred_rf))
print("RF ROC-AUC:", roc_auc_score(y_test, y_prob_rf))
print(classification_report(y_test, y_pred_rf))

RF Accuracy: 0.8883876745497363
RF ROC-AUC: 0.7775284563232941
              precision    recall  f1-score   support

         0.0       0.90      0.99      0.94     26918
         1.0       0.30      0.03      0.06      3231

    accuracy                           0.89     30149
   macro avg       0.60      0.51      0.50     30149
weighted avg       0.83      0.89      0.85     30149



In [18]:
import numpy as np

# Try lower threshold
threshold = 0.3

y_pred_adj = (y_prob >= threshold).astype(int)

print("Adjusted Threshold:", threshold)
print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print(classification_report(y_test, y_pred_adj))

Adjusted Threshold: 0.3
ROC-AUC: 0.7174855975007515
              precision    recall  f1-score   support

         0.0       0.96      0.36      0.53     26918
         1.0       0.14      0.89      0.25      3231

    accuracy                           0.42     30149
   macro avg       0.55      0.62      0.39     30149
weighted avg       0.88      0.42      0.50     30149



In [19]:
import numpy as np

# Try lower threshold
threshold = 0.2

y_pred_adj = (y_prob >= threshold).astype(int)

print("Adjusted Threshold:", threshold)
print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print(classification_report(y_test, y_pred_adj))

Adjusted Threshold: 0.2
ROC-AUC: 0.7174855975007515
              precision    recall  f1-score   support

         0.0       0.97      0.10      0.19     26918
         1.0       0.12      0.98      0.21      3231

    accuracy                           0.20     30149
   macro avg       0.54      0.54      0.20     30149
weighted avg       0.88      0.20      0.19     30149



In [20]:
patient_hours = icu_hourly.groupby("patientunitstayid")["hour"].max()

print("Total patients:", len(patient_hours))
print("Patients with >=24h:", (patient_hours >= 24).sum())

Total patients: 2468
Patients with >=24h: 1927


In [21]:
# Patients with at least 24 hours
valid_patients = patient_hours[patient_hours >= 24].index

icu_24 = icu_hourly[
    icu_hourly["patientunitstayid"].isin(valid_patients)
]

print("Rows:", icu_24.shape)
print("Patients:", icu_24["patientunitstayid"].nunique())

Rows: (142752, 16)
Patients: 1927


In [22]:
icu_24 = icu_24[icu_24["hour"] < 24]
print(icu_24["hour"].max())

23


In [23]:
icu_24 = icu_24.merge(
    patients[["patientunitstayid", "mortality"]],
    on="patientunitstayid",
    how="left"
)

print(icu_24["mortality"].isna().sum())

577


In [24]:
icu_24 = icu_24.dropna(subset=["mortality"])

print("Rows after drop:", icu_24.shape)
print("Unique patients:", icu_24["patientunitstayid"].nunique())

Rows after drop: (44847, 17)
Unique patients: 1899


In [25]:
hours_per_patient = icu_24.groupby("patientunitstayid")["hour"].count()

print("Patients with exactly 24 rows:",
      (hours_per_patient == 24).sum())

print("Patients with less than 24 rows:",
      (hours_per_patient < 24).sum())

Patients with exactly 24 rows: 342
Patients with less than 24 rows: 478


In [26]:
print(icu_hourly.shape)
print(icu_hourly["patientunitstayid"].nunique())

(149775, 16)
2468


In [27]:
icu_24 = icu_hourly[icu_hourly["hour"] < 24]

In [28]:
hours_per_patient = (
    icu_24.groupby("patientunitstayid")["hour"]
    .nunique()
)

print("Patients with 24 distinct hours:",
      (hours_per_patient == 24).sum())

Patients with 24 distinct hours: 367


In [29]:
valid_patients = hours_per_patient[hours_per_patient == 24].index

icu_seq = icu_24[
    icu_24["patientunitstayid"].isin(valid_patients)
]

print("Rows:", icu_seq.shape)
print("Patients:", icu_seq["patientunitstayid"].nunique())

Rows: (8808, 16)
Patients: 367


In [30]:
icu_seq = icu_seq.merge(
    patients[["patientunitstayid", "mortality"]],
    on="patientunitstayid",
    how="left"
)

print("Missing mortality:", icu_seq["mortality"].isna().sum())

Missing mortality: 72


In [31]:
icu_seq = icu_seq.dropna(subset=["mortality"])

print("Rows after drop:", icu_seq.shape)
print("Patients after drop:", icu_seq["patientunitstayid"].nunique())

Rows after drop: (8736, 17)
Patients after drop: 364


In [32]:
hours_check = icu_seq.groupby("patientunitstayid")["hour"].nunique()

print("Patients with full 24h:",
      (hours_check == 24).sum())

print("Any incomplete left:",
      (hours_check < 24).sum())

Patients with full 24h: 364
Any incomplete left: 0


In [33]:
icu_seq = icu_seq.sort_values(["patientunitstayid", "hour"])


In [34]:
feature_cols = [
    col for col in icu_seq.columns
    if col not in ["patientunitstayid", "hour", "mortality"]
]

print("Feature count:", len(feature_cols))

Feature count: 14


In [35]:
import numpy as np

patients_unique = icu_seq["patientunitstayid"].unique()

X_list = []
y_list = []

for pid in patients_unique:
    patient_data = icu_seq[icu_seq["patientunitstayid"] == pid]
    
    X_list.append(patient_data[feature_cols].values)
    y_list.append(patient_data["mortality"].iloc[0])

X_seq = np.array(X_list)
y_seq = np.array(y_list)

print("X shape:", X_seq.shape)
print("y shape:", y_seq.shape)

X shape: (364, 24, 14)
y shape: (364,)


In [36]:
import os
os.getcwd()

'E:\\icu_project'

In [37]:
os.listdir()

['.ipynb_checkpoints',
 '01_ICU_Mortality_Baseline.ipynb.ipynb',
 'anaconda_projects',
 'data',
 'models',
 'step1_data_loading.ipynb',
 'Untitled.ipynb']

In [38]:
for root, dirs, files in os.walk("E:/"):
    for file in files:
        if "processed_icu_hourly_v1.csv" in file:
            print(os.path.join(root, file))

E:/icu_project\data\processed_icu_hourly_v1.csv


In [39]:
import pandas as pd

df = pd.read_csv(r"E:\icu_project\data\processed_icu_hourly_v1.csv")

In [40]:
print(df.shape)
print(df.columns)

(149775, 16)
Index(['patientunitstayid', 'hour', 'sao2', 'heartrate', 'respiration', 'BUN',
       'HCO3', 'Hct', 'Hgb', 'WBC x 1000', 'creatinine', 'magnesium', 'pH',
       'platelets x 1000', 'potassium', 'sodium'],
      dtype='object')


In [41]:
print(df.columns)

Index(['patientunitstayid', 'hour', 'sao2', 'heartrate', 'respiration', 'BUN',
       'HCO3', 'Hct', 'Hgb', 'WBC x 1000', 'creatinine', 'magnesium', 'pH',
       'platelets x 1000', 'potassium', 'sodium'],
      dtype='object')


In [42]:
import os

for root, dirs, files in os.walk(r"E:\icu_project"):
    for file in files:
        if "patient" in file.lower():
            print(os.path.join(root, file))

E:\icu_project\data\eicu_demo\apachePatientResult.csv
E:\icu_project\data\eicu_demo\patient.csv


In [43]:
import pandas as pd

patient_df = pd.read_csv(r"E:\icu_project\data\eicu_demo\patient.csv")

In [44]:
print(patient_df.columns)

Index(['patientunitstayid', 'patienthealthsystemstayid', 'gender', 'age',
       'ethnicity', 'hospitalid', 'wardid', 'apacheadmissiondx',
       'admissionheight', 'hospitaladmittime24', 'hospitaladmitoffset',
       'hospitaladmitsource', 'hospitaldischargeyear',
       'hospitaldischargetime24', 'hospitaldischargeoffset',
       'hospitaldischargelocation', 'hospitaldischargestatus', 'unittype',
       'unitadmittime24', 'unitadmitsource', 'unitvisitnumber', 'unitstaytype',
       'admissionweight', 'dischargeweight', 'unitdischargetime24',
       'unitdischargeoffset', 'unitdischargelocation', 'unitdischargestatus',
       'uniquepid'],
      dtype='object')


In [45]:
patient_df = patient_df[[
    "patientunitstayid",
    "hospitaldischargestatus"
]]

patient_df["mortality"] = (
    patient_df["hospitaldischargestatus"] == "Expired"
).astype(int)

patient_df = patient_df[[
    "patientunitstayid",
    "mortality"
]]

In [46]:
print(patient_df["mortality"].value_counts())

mortality
0    2308
1     212
Name: count, dtype: int64


In [47]:
df = df.merge(
    patient_df,
    on="patientunitstayid",
    how="left"
)

In [48]:
print(df["mortality"].isna().sum())
print(df["mortality"].value_counts())

0
mortality
0    131504
1     18271
Name: count, dtype: int64


In [49]:
df.to_csv(
    r"E:\icu_project\data\processed_icu_hourly_v2.csv",
    index=False
)

In [50]:
import numpy as np

feature_cols = [
    "heartrate","respiration","sao2","BUN","creatinine",
    "sodium","potassium","magnesium",
    "Hct","Hgb","WBC x 1000","platelets x 1000",
    "HCO3","pH"
]

patients = []

for pid, group in df.groupby("patientunitstayid"):
    group = group.sort_values("hour")
    if len(group) == 24 and set(group["hour"]) == set(range(24)):
        patients.append(group)

df_24 = pd.concat(patients)

X = []
y = []

for pid, group in df_24.groupby("patientunitstayid"):
    X.append(group[feature_cols].values)
    y.append(group["mortality"].iloc[0])

X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Mortality rate:", y.mean())

X shape: (5, 24, 14)
y shape: (5,)
Mortality rate: 0.2


In [51]:
import os
os.listdir(r"E:\icu_project\data\challenge2012")

['Outcomes-a.txt',
 'Outcomes-b.txt',
 'Outcomes-c.txt',
 'set-a',
 'set-b',
 'set-c']

In [52]:
import pandas as pd

outcomes_a = pd.read_csv(
    r"E:\icu_project\data\challenge2012\Outcomes-a.txt"
)

outcomes_b = pd.read_csv(
    r"E:\icu_project\data\challenge2012\Outcomes-b.txt"
)

outcomes_c = pd.read_csv(
    r"E:\icu_project\data\challenge2012\Outcomes-c.txt"
)

outcomes = pd.concat([outcomes_a, outcomes_b, outcomes_c])

print(outcomes.head())
print(outcomes.columns)

   RecordID  SAPS-I  SOFA  Length_of_stay  Survival  In-hospital_death
0    132539       6     1               5        -1                  0
1    132540      16     8               8        -1                  0
2    132541      21    11              19        -1                  0
3    132543       7     1               9       575                  0
4    132545      17     2               4       918                  0
Index(['RecordID', 'SAPS-I', 'SOFA', 'Length_of_stay', 'Survival',
       'In-hospital_death'],
      dtype='object')


In [53]:
print(outcomes.columns)
print(outcomes.head())

Index(['RecordID', 'SAPS-I', 'SOFA', 'Length_of_stay', 'Survival',
       'In-hospital_death'],
      dtype='object')
   RecordID  SAPS-I  SOFA  Length_of_stay  Survival  In-hospital_death
0    132539       6     1               5        -1                  0
1    132540      16     8               8        -1                  0
2    132541      21    11              19        -1                  0
3    132543       7     1               9       575                  0
4    132545      17     2               4       918                  0


In [54]:
outcomes = outcomes[["RecordID", "In-hospital_death"]]
outcomes.columns = ["patient_id", "mortality"]

print("Mortality rate:", outcomes["mortality"].mean())
print("Total patients:", len(outcomes))

Mortality rate: 0.14225
Total patients: 12000


In [55]:
import os

set_a_path = r"E:\icu_project\data\challenge2012\set-a"
print(os.listdir(set_a_path)[:5])

['132539.txt', '132540.txt', '132541.txt', '132543.txt', '132545.txt']


In [56]:
sample_file = os.listdir(set_a_path)[0]

df_sample = pd.read_csv(
    rf"E:\icu_project\data\challenge2012\set-a\{sample_file}"
)

print(df_sample.head())
print(df_sample.columns)

    Time Parameter     Value
0  00:00  RecordID  132539.0
1  00:00       Age      54.0
2  00:00    Gender       0.0
3  00:00    Height      -1.0
4  00:00   ICUType       4.0
Index(['Time', 'Parameter', 'Value'], dtype='object')


In [57]:
import os
import pandas as pd

base_path = r"E:\icu_project\data\challenge2012"
sets = ["set-a", "set-b", "set-c"]

all_params = set()

for s in sets:
    folder = os.path.join(base_path, s)
    for file in os.listdir(folder):
        df_temp = pd.read_csv(os.path.join(folder, file))
        
        # Drop missing parameters
        params = df_temp["Parameter"].dropna()
        
        # Convert explicitly to string
        params = params.astype(str)
        
        all_params.update(params.unique())

all_params = sorted(list(all_params))

print("Total variables:", len(all_params))

Total variables: 42


In [58]:
mortality_dict = dict(
    zip(outcomes["patient_id"], outcomes["mortality"])
)

In [59]:
param_index = {p: i for i, p in enumerate(all_params)}
num_vars = len(all_params)

In [60]:
import numpy as np

X_list = []
mask_list = []
y_list = []

for s in sets:
    folder = os.path.join(base_path, s)
    
    for file in os.listdir(folder):
        patient_id = int(file.split(".")[0])
        
        df = pd.read_csv(os.path.join(folder, file))
        
        # Clean
        df = df.dropna(subset=["Parameter", "Value"])
        df["Parameter"] = df["Parameter"].astype(str)
        
        # Extract hour
        df["hour"] = df["Time"].str.split(":").str[0].astype(int)
        df = df[df["hour"] < 24]
        
        patient_matrix = np.zeros((24, num_vars))
        patient_mask = np.zeros((24, num_vars))
        
        for h, param, val in zip(df["hour"], df["Parameter"], df["Value"]):
            if param in param_index:
                idx = param_index[param]
                patient_matrix[h, idx] = val
                patient_mask[h, idx] = 1
        
        X_list.append(patient_matrix)
        mask_list.append(patient_mask)
        
        y_list.append(mortality_dict.get(patient_id, 0))

In [61]:
X = np.array(X_list)
mask = np.array(mask_list)
y = np.array(y_list)

print("X shape:", X.shape)
print("Mask shape:", mask.shape)
print("y shape:", y.shape)
print("Mortality rate:", y.mean())

X shape: (12000, 24, 42)
Mask shape: (12000, 24, 42)
y shape: (12000,)
Mortality rate: 0.14225


In [62]:
import numpy as np

means = np.zeros(X.shape[2])
stds = np.zeros(X.shape[2])

for v in range(X.shape[2]):
    values = X[:, :, v][mask[:, :, v] == 1]
    means[v] = values.mean()
    stds[v] = values.std() + 1e-6

In [63]:
X_norm = X.copy()

for v in range(X.shape[2]):
    X_norm[:, :, v][mask[:, :, v] == 1] = (
        (X[:, :, v][mask[:, :, v] == 1] - means[v]) / stds[v]
    )

In [64]:
X_final = np.concatenate([X_norm, mask], axis=2)

print("Final shape:", X_final.shape)

Final shape: (12000, 24, 84)


In [65]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_final,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [66]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
X_test_t  = torch.tensor(X_test, dtype=torch.float32).to(device)

y_train_t = torch.tensor(y_train, dtype=torch.float32).to(device)
y_test_t  = torch.tensor(y_test, dtype=torch.float32).to(device)

In [67]:
import torch.nn as nn

class ICU_LSTM(nn.Module):
    def __init__(self, input_dim=84, hidden_dim=128, num_layers=2):
        super().__init__()
        
        self.lstm = nn.LSTM(
            input_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.3
        )
        
        self.fc = nn.Linear(hidden_dim, 1)
        
    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]   # last time step
        out = self.fc(out)
        return out.squeeze()

In [68]:
model = ICU_LSTM().to(device)

In [69]:
import numpy as np

pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()
pos_weight = torch.tensor(pos_weight, dtype=torch.float32).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

In [70]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [71]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

epochs = 20

for epoch in range(epochs):
    model.train()
    total_loss = 0
    
    for xb, yb in train_loader:
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

Epoch 1, Loss: 1.0454
Epoch 2, Loss: 0.9433
Epoch 3, Loss: 0.8943
Epoch 4, Loss: 0.8764
Epoch 5, Loss: 0.8531
Epoch 6, Loss: 0.8335
Epoch 7, Loss: 0.8145
Epoch 8, Loss: 0.8035
Epoch 9, Loss: 0.7736
Epoch 10, Loss: 0.7545
Epoch 11, Loss: 0.7195
Epoch 12, Loss: 0.7094
Epoch 13, Loss: 0.6705
Epoch 14, Loss: 0.6326
Epoch 15, Loss: 0.6337
Epoch 16, Loss: 0.5781
Epoch 17, Loss: 0.5953
Epoch 18, Loss: 0.5225
Epoch 19, Loss: 0.5027
Epoch 20, Loss: 0.4606


In [72]:
from sklearn.metrics import roc_auc_score

model.eval()
with torch.no_grad():
    logits = model(X_test_t)
    probs = torch.sigmoid(logits).cpu().numpy()

roc = roc_auc_score(y_test, probs)
print("Test ROC-AUC:", roc)

Test ROC-AUC: 0.7740796075878875


In [73]:
import torch.nn as nn

class ICU_BiLSTM(nn.Module):
    def __init__(self, input_dim=84, hidden_dim=128, num_layers=2):
        super().__init__()
        
        self.lstm = nn.LSTM(
            input_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.3,
            bidirectional=True
        )
        
        self.fc = nn.Linear(hidden_dim * 2, 1)
        
    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        out = self.fc(out)
        return out.squeeze()

In [74]:
model = ICU_BiLSTM().to(device)

In [75]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [76]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

epochs = 20

for epoch in range(epochs):
    model.train()
    total_loss = 0
    
    for xb, yb in train_loader:
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

Epoch 1, Loss: 1.0162
Epoch 2, Loss: 0.9337
Epoch 3, Loss: 0.8915
Epoch 4, Loss: 0.8831
Epoch 5, Loss: 0.8483
Epoch 6, Loss: 0.8228
Epoch 7, Loss: 0.8120
Epoch 8, Loss: 0.7846
Epoch 9, Loss: 0.7805
Epoch 10, Loss: 0.7586
Epoch 11, Loss: 0.7173
Epoch 12, Loss: 0.6999
Epoch 13, Loss: 0.6725
Epoch 14, Loss: 0.6356
Epoch 15, Loss: 0.6133
Epoch 16, Loss: 0.5837
Epoch 17, Loss: 0.5402
Epoch 18, Loss: 0.5181
Epoch 19, Loss: 0.4854
Epoch 20, Loss: 0.4495


In [77]:
from sklearn.metrics import roc_auc_score

model.eval()
with torch.no_grad():
    logits = model(X_test_t)
    probs = torch.sigmoid(logits).cpu().numpy()

roc = roc_auc_score(y_test, probs)
print("Test ROC-AUC:", roc)

Test ROC-AUC: 0.7769437944280102


In [78]:
np.save(r"E:\icu_project\data\X_24h.npy", X_final)
np.save(r"E:\icu_project\data\y_24h.npy", y)


In [79]:
np.save(r"E:\icu_project\data\means_24h.npy", means)
np.save(r"E:\icu_project\data\stds_24h.npy", stds)

In [80]:
torch.save(model.state_dict(), r"E:\icu_project\models\lstm_24h.pt")

In [81]:
print(X_final.shape)
print(y.shape)

(12000, 24, 84)
(12000,)


In [82]:
import os
import numpy as np
import pandas as pd

base_path = r"E:\icu_project\data\challenge2012"
sets = ["set-a","set-b","set-c"]

In [83]:
outcomes_a = pd.read_csv(r"E:\icu_project\data\challenge2012\Outcomes-a.txt")
outcomes_b = pd.read_csv(r"E:\icu_project\data\challenge2012\Outcomes-b.txt")
outcomes_c = pd.read_csv(r"E:\icu_project\data\challenge2012\Outcomes-c.txt")

outcomes = pd.concat([outcomes_a,outcomes_b,outcomes_c])
outcomes = outcomes[["RecordID","In-hospital_death"]]
outcomes.columns = ["patient_id","mortality"]

mortality_dict = dict(zip(outcomes["patient_id"],outcomes["mortality"]))

In [84]:
all_params = set()

for s in sets:
    folder = os.path.join(base_path,s)

    for file in os.listdir(folder):
        df_temp = pd.read_csv(os.path.join(folder,file))

        params = df_temp["Parameter"].dropna().astype(str)

        all_params.update(params.unique())

all_params = sorted(list(all_params))
param_index = {p:i for i,p in enumerate(all_params)}
num_vars = len(all_params)

print("variables:",num_vars)

variables: 42


In [85]:
X_list = []
mask_list = []
y_list = []

for s in sets:

    folder = os.path.join(base_path,s)

    for file in os.listdir(folder):

        patient_id = int(file.split(".")[0])

        df = pd.read_csv(os.path.join(folder,file))

        df = df.dropna(subset=["Parameter","Value"])
        df["Parameter"] = df["Parameter"].astype(str)

        df["hour"] = df["Time"].str.split(":").str[0].astype(int)

        df = df[df["hour"] < 48]

        patient_matrix = np.zeros((48,num_vars))
        patient_mask   = np.zeros((48,num_vars))

        for h,param,val in zip(df["hour"],df["Parameter"],df["Value"]):

            if param in param_index:

                idx = param_index[param]

                patient_matrix[h,idx] = val
                patient_mask[h,idx] = 1

        X_list.append(patient_matrix)
        mask_list.append(patient_mask)

        y_list.append(mortality_dict.get(patient_id,0))

In [86]:
X = np.array(X_list)
mask = np.array(mask_list)
y = np.array(y_list)

print("X:",X.shape)
print("mask:",mask.shape)
print("y:",y.shape)
print("mortality:",y.mean())

X: (12000, 48, 42)
mask: (12000, 48, 42)
y: (12000,)
mortality: 0.14225


In [87]:
means = np.zeros(X.shape[2])
stds  = np.zeros(X.shape[2])

for v in range(X.shape[2]):
    values = X[:, :, v][mask[:, :, v] == 1]
    means[v] = values.mean()
    stds[v]  = values.std() + 1e-6

In [88]:
X_norm = X.copy()

for v in range(X.shape[2]):
    observed = mask[:, :, v] == 1
    X_norm[:, :, v][observed] = (X[:, :, v][observed] - means[v]) / stds[v]

In [89]:
X_final = np.concatenate([X_norm, mask], axis=2)

print(X_final.shape)

(12000, 48, 84)


In [90]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_final,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [91]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
X_test_t  = torch.tensor(X_test, dtype=torch.float32).to(device)

y_train_t = torch.tensor(y_train, dtype=torch.float32).to(device)
y_test_t  = torch.tensor(y_test, dtype=torch.float32).to(device)

In [92]:
import torch.nn as nn

class ICU_LSTM(nn.Module):
    
    def __init__(self, input_dim=84, hidden_dim=128, num_layers=2):
        super().__init__()
        
        self.lstm = nn.LSTM(
            input_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.3
        )
        
        self.fc = nn.Linear(hidden_dim,1)
        
    def forward(self,x):
        
        out,_ = self.lstm(x)
        
        out = out[:,-1,:]      # last time step
        
        out = self.fc(out)
        
        return out.squeeze()

In [93]:
model = ICU_LSTM().to(device)

In [94]:
import numpy as np

pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()

pos_weight = torch.tensor(pos_weight,dtype=torch.float32).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

In [95]:
optimizer = torch.optim.Adam(model.parameters(),lr=1e-3)

In [96]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_t,y_train_t)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

In [97]:
epochs = 25

for epoch in range(epochs):

    model.train()
    total_loss = 0

    for xb,yb in train_loader:

        optimizer.zero_grad()

        logits = model(xb)

        loss = criterion(logits,yb)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss {total_loss/len(train_loader):.4f}")

Epoch 1 Loss 0.9783
Epoch 2 Loss 0.8767
Epoch 3 Loss 0.8482
Epoch 4 Loss 0.8383
Epoch 5 Loss 0.8249
Epoch 6 Loss 0.7990
Epoch 7 Loss 0.7817
Epoch 8 Loss 0.7745
Epoch 9 Loss 0.7467
Epoch 10 Loss 0.7537
Epoch 11 Loss 0.7173
Epoch 12 Loss 0.7011
Epoch 13 Loss 0.6822
Epoch 14 Loss 0.6658
Epoch 15 Loss 0.6765
Epoch 16 Loss 0.6441
Epoch 17 Loss 0.6240
Epoch 18 Loss 0.6200
Epoch 19 Loss 0.5705
Epoch 20 Loss 0.5287
Epoch 21 Loss 0.5527
Epoch 22 Loss 0.5284
Epoch 23 Loss 0.4871
Epoch 24 Loss 0.4721
Epoch 25 Loss 0.4160


In [98]:
from sklearn.metrics import roc_auc_score
import torch

model.eval()

with torch.no_grad():
    
    logits = model(X_test_t)
    
    probs = torch.sigmoid(logits).cpu().numpy()

roc = roc_auc_score(y_test,probs)

print("Test ROC-AUC:",roc)

Test ROC-AUC: 0.8012245787395014


In [99]:
import torch
import torch.nn as nn

class AttentionLSTM(nn.Module):

    def __init__(self,input_dim=84,hidden_dim=128,num_layers=2):

        super().__init__()

        self.lstm = nn.LSTM(
            input_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.3
        )

        self.attention = nn.Linear(hidden_dim,1)

        self.fc = nn.Linear(hidden_dim,1)

    def forward(self,x):

        out,_ = self.lstm(x)

        attn_weights = torch.softmax(self.attention(out),dim=1)

        context = torch.sum(attn_weights * out,dim=1)

        out = self.fc(context)

        return out.squeeze()

In [101]:
model = AttentionLSTM().to(device)

In [102]:
import torch
import numpy as np
import torch.nn as nn

pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()
pos_weight = torch.tensor(pos_weight, dtype=torch.float32).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

In [103]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [104]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_t, y_train_t)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

In [105]:
epochs = 25

for epoch in range(epochs):

    model.train()
    total_loss = 0

    for xb, yb in train_loader:

        optimizer.zero_grad()

        logits = model(xb)

        loss = criterion(logits, yb)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss {total_loss/len(train_loader):.4f}")

Epoch 1, Loss 0.9887
Epoch 2, Loss 0.8780
Epoch 3, Loss 0.8611
Epoch 4, Loss 0.8288
Epoch 5, Loss 0.7953
Epoch 6, Loss 0.7912
Epoch 7, Loss 0.7687
Epoch 8, Loss 0.7582
Epoch 9, Loss 0.7396
Epoch 10, Loss 0.7211
Epoch 11, Loss 0.7011
Epoch 12, Loss 0.6837
Epoch 13, Loss 0.6611
Epoch 14, Loss 0.6457
Epoch 15, Loss 0.6208
Epoch 16, Loss 0.5941
Epoch 17, Loss 0.5727
Epoch 18, Loss 0.5431
Epoch 19, Loss 0.5306
Epoch 20, Loss 0.4904
Epoch 21, Loss 0.4948
Epoch 22, Loss 0.4299
Epoch 23, Loss 0.4065
Epoch 24, Loss 0.3747
Epoch 25, Loss 0.3462


In [106]:
from sklearn.metrics import roc_auc_score

model.eval()

with torch.no_grad():

    logits = model(X_test_t)

    probs = torch.sigmoid(logits).cpu().numpy()

roc = roc_auc_score(y_test, probs)

print("Test ROC-AUC:", roc)

Test ROC-AUC: 0.7836079069217611


In [107]:
feature_means = np.zeros(X.shape[2])

for v in range(X.shape[2]):
    values = X[:,:,v][mask[:,:,v]==1]
    feature_means[v] = values.mean()

In [108]:
feature_means = torch.tensor(feature_means, dtype=torch.float32).to(device)

In [109]:
import torch
import torch.nn as nn

class GRUD(nn.Module):

    def __init__(self, input_dim=42, hidden_dim=128):
        super().__init__()

        self.hidden_dim = hidden_dim

        self.gru = nn.GRUCell(input_dim, hidden_dim)

        self.gamma_x = nn.Linear(input_dim, input_dim)
        self.gamma_h = nn.Linear(input_dim, hidden_dim)

        self.fc = nn.Linear(hidden_dim,1)

    def forward(self,x,mask,mean):

        batch,seq,dim = x.shape

        h = torch.zeros(batch,self.hidden_dim).to(x.device)

        x_last = mean.unsqueeze(0).repeat(batch,1)

        for t in range(seq):

            m = mask[:,t,:]

            gamma_x = torch.exp(-torch.relu(self.gamma_x(m)))
            gamma_h = torch.exp(-torch.relu(self.gamma_h(m)))

            x_t = m * x[:,t,:] + (1-m) * (gamma_x * x_last + (1-gamma_x) * mean)

            h = gamma_h * h

            h = self.gru(x_t,h)

            x_last = x_t

        out = self.fc(h)

        return out.squeeze()

In [110]:
model = GRUD(input_dim=42).to(device)

In [111]:
from sklearn.model_selection import train_test_split

X_train, X_test, M_train, M_test, y_train, y_test = train_test_split(
    X_norm,
    mask,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [112]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
X_test_t  = torch.tensor(X_test, dtype=torch.float32).to(device)

M_train_t = torch.tensor(M_train, dtype=torch.float32).to(device)
M_test_t  = torch.tensor(M_test, dtype=torch.float32).to(device)

y_train_t = torch.tensor(y_train, dtype=torch.float32).to(device)
y_test_t  = torch.tensor(y_test, dtype=torch.float32).to(device)

In [113]:
import numpy as np

feature_means = np.zeros(X_norm.shape[2])

for v in range(X_norm.shape[2]):
    vals = X_norm[:,:,v][mask[:,:,v]==1]
    feature_means[v] = vals.mean()

feature_means = torch.tensor(feature_means, dtype=torch.float32).to(device)

In [114]:
import torch.nn as nn

class GRUD(nn.Module):

    def __init__(self, input_dim=42, hidden_dim=128):
        super().__init__()

        self.hidden_dim = hidden_dim

        self.gru_cell = nn.GRUCell(input_dim, hidden_dim)

        self.gamma_x = nn.Linear(input_dim, input_dim)
        self.gamma_h = nn.Linear(input_dim, hidden_dim)

        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x, m, mean):

        batch, seq, dim = x.shape

        h = torch.zeros(batch, self.hidden_dim).to(x.device)

        x_last = mean.unsqueeze(0).repeat(batch,1)

        for t in range(seq):

            m_t = m[:,t,:]

            gamma_x = torch.exp(-torch.relu(self.gamma_x(m_t)))
            gamma_h = torch.exp(-torch.relu(self.gamma_h(m_t)))

            x_t = m_t * x[:,t,:] + (1-m_t) * (gamma_x * x_last + (1-gamma_x) * mean)

            h = gamma_h * h

            h = self.gru_cell(x_t, h)

            x_last = x_t

        out = self.fc(h)

        return out.squeeze()

In [115]:
model = GRUD(input_dim=42).to(device)

In [116]:
pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()

pos_weight = torch.tensor(pos_weight, dtype=torch.float32).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

In [117]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [118]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_t, M_train_t, y_train_t)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

In [119]:
epochs = 25

for epoch in range(epochs):

    model.train()
    total_loss = 0

    for xb, mb, yb in train_loader:

        optimizer.zero_grad()

        logits = model(xb, mb, feature_means)

        loss = criterion(logits, yb)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss {total_loss/len(train_loader):.4f}")

Epoch 1 Loss 0.8780
Epoch 2 Loss 0.8136
Epoch 3 Loss 0.7923
Epoch 4 Loss 0.7751
Epoch 5 Loss 0.7567
Epoch 6 Loss 0.7311
Epoch 7 Loss 0.7085
Epoch 8 Loss 0.6819
Epoch 9 Loss 0.6528
Epoch 10 Loss 0.6265
Epoch 11 Loss 0.6030
Epoch 12 Loss 0.5597
Epoch 13 Loss 0.5175
Epoch 14 Loss 0.4763
Epoch 15 Loss 0.4498
Epoch 16 Loss 0.4132
Epoch 17 Loss 0.3500
Epoch 18 Loss 0.3345
Epoch 19 Loss 0.3035
Epoch 20 Loss 0.2662
Epoch 21 Loss 0.2696
Epoch 22 Loss 0.1968
Epoch 23 Loss 0.2117
Epoch 24 Loss 0.2592
Epoch 25 Loss 0.1530


In [120]:
from sklearn.metrics import roc_auc_score

model.eval()

with torch.no_grad():

    logits = model(X_test_t, M_test_t, feature_means)

    probs = torch.sigmoid(logits).cpu().numpy()

roc = roc_auc_score(y_test, probs)

print("Test ROC-AUC:", roc)

Test ROC-AUC: 0.8113738554290656


In [122]:
delta = np.zeros_like(mask)

for i in range(mask.shape[0]):
    for d in range(mask.shape[2]):

        for t in range(1, mask.shape[1]):

            if mask[i,t,d] == 1:
                delta[i,t,d] = 1
            else:
                delta[i,t,d] = delta[i,t-1,d] + 1

In [123]:
X_train, X_test, M_train, M_test, D_train, D_test, y_train, y_test = train_test_split(
    X_norm,
    mask,
    delta,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [124]:
D_train_t = torch.tensor(D_train, dtype=torch.float32).to(device)
D_test_t  = torch.tensor(D_test, dtype=torch.float32).to(device)

In [125]:
train_dataset = TensorDataset(X_train_t, M_train_t, D_train_t, y_train_t)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

In [128]:
import torch
import torch.nn as nn

class GRUD(nn.Module):

    def __init__(self, input_dim=42, hidden_dim=128):
        super().__init__()

        self.hidden_dim = hidden_dim

        self.gru_cell = nn.GRUCell(input_dim, hidden_dim)

        self.gamma_x = nn.Linear(input_dim, input_dim)
        self.gamma_h = nn.Linear(input_dim, hidden_dim)

        self.fc = nn.Linear(hidden_dim,1)

    def forward(self, x, m, d, mean):

        batch, seq, dim = x.shape

        h = torch.zeros(batch, self.hidden_dim).to(x.device)

        x_last = mean.unsqueeze(0).repeat(batch,1)

        for t in range(seq):

            m_t = m[:,t,:]
            d_t = d[:,t,:]

            gamma_x = torch.exp(-torch.relu(self.gamma_x(d_t)))
            gamma_h = torch.exp(-torch.relu(self.gamma_h(d_t)))

            x_t = m_t * x[:,t,:] + (1-m_t) * (gamma_x * x_last + (1-gamma_x) * mean)

            h = gamma_h * h

            h = self.gru_cell(x_t, h)

            x_last = x_t

        out = self.fc(h)

        return out.squeeze()

In [129]:
model = GRUD(input_dim=42).to(device)

In [130]:
epochs = 25

for epoch in range(epochs):

    model.train()
    total_loss = 0

    for xb, mb, db, yb in train_loader:

        optimizer.zero_grad()

        logits = model(xb, mb, db, feature_means)

        loss = criterion(logits, yb)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss {total_loss/len(train_loader):.4f}")

Epoch 1 Loss 1.1888
Epoch 2 Loss 1.1888
Epoch 3 Loss 1.1888
Epoch 4 Loss 1.1888
Epoch 5 Loss 1.1888
Epoch 6 Loss 1.1888
Epoch 7 Loss 1.1888
Epoch 8 Loss 1.1888
Epoch 9 Loss 1.1888
Epoch 10 Loss 1.1888
Epoch 11 Loss 1.1888
Epoch 12 Loss 1.1888
Epoch 13 Loss 1.1888
Epoch 14 Loss 1.1888
Epoch 15 Loss 1.1888
Epoch 16 Loss 1.1888
Epoch 17 Loss 1.1888
Epoch 18 Loss 1.1888
Epoch 19 Loss 1.1888
Epoch 20 Loss 1.1888
Epoch 21 Loss 1.1888
Epoch 22 Loss 1.1888
Epoch 23 Loss 1.1888
Epoch 24 Loss 1.1888
Epoch 25 Loss 1.1888


In [131]:
from sklearn.metrics import roc_auc_score

model.eval()

with torch.no_grad():

    logits = model(X_test_t, M_test_t, D_test_t, feature_means)

    probs = torch.sigmoid(logits).cpu().numpy()

roc = roc_auc_score(y_test, probs)

print("Test ROC-AUC:", roc)

Test ROC-AUC: 0.5342520285022909


In [132]:
delta = np.zeros_like(mask)

for i in range(mask.shape[0]):
    for d in range(mask.shape[2]):

        delta[i,0,d] = 0

        for t in range(1, mask.shape[1]):

            if mask[i,t-1,d] == 1:
                delta[i,t,d] = 1
            else:
                delta[i,t,d] = delta[i,t-1,d] + 1

In [133]:
print(delta.shape)

(12000, 48, 42)


In [134]:
X_train, X_test, M_train, M_test, D_train, D_test, y_train, y_test = train_test_split(
    X_norm,
    mask,
    delta,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [135]:
D_train_t = torch.tensor(D_train, dtype=torch.float32).to(device)
D_test_t  = torch.tensor(D_test, dtype=torch.float32).to(device)

In [136]:
train_dataset = TensorDataset(X_train_t, M_train_t, D_train_t, y_train_t)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

In [137]:
model = GRUD(input_dim=42).to(device)

In [138]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [139]:
epochs = 25

for epoch in range(epochs):

    model.train()
    total_loss = 0

    for xb, mb, db, yb in train_loader:

        optimizer.zero_grad()

        logits = model(xb, mb, db, feature_means)

        loss = criterion(logits, yb)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss {total_loss/len(train_loader):.4f}")

Epoch 1 Loss 0.9229
Epoch 2 Loss 0.8452
Epoch 3 Loss 0.8238
Epoch 4 Loss 0.8104
Epoch 5 Loss 0.8034
Epoch 6 Loss 0.7893
Epoch 7 Loss 0.7687
Epoch 8 Loss 0.7597
Epoch 9 Loss 0.7478
Epoch 10 Loss 0.7257
Epoch 11 Loss 0.7270
Epoch 12 Loss 0.7051
Epoch 13 Loss 0.6759
Epoch 14 Loss 0.6598
Epoch 15 Loss 0.6437
Epoch 16 Loss 0.6206
Epoch 17 Loss 0.5897
Epoch 18 Loss 0.5677
Epoch 19 Loss 0.5527
Epoch 20 Loss 0.5254
Epoch 21 Loss 0.4976
Epoch 22 Loss 0.4655
Epoch 23 Loss 0.4459
Epoch 24 Loss 0.4260
Epoch 25 Loss 0.4130


In [140]:
from sklearn.metrics import roc_auc_score

model.eval()

with torch.no_grad():

    logits = model(X_test_t, M_test_t, D_test_t, feature_means)

    probs = torch.sigmoid(logits).cpu().numpy()

roc = roc_auc_score(y_test, probs)

print("Test ROC-AUC:", roc)

Test ROC-AUC: 0.8168145285913071


In [141]:
model = GRUD(input_dim=42, hidden_dim=256).to(device)

In [142]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-5
)

In [143]:
epochs = 25

for epoch in range(epochs):

    model.train()
    total_loss = 0

    for xb, mb, db, yb in train_loader:

        optimizer.zero_grad()

        logits = model(xb, mb, db, feature_means)

        loss = criterion(logits, yb)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss {total_loss/len(train_loader):.4f}")

Epoch 1 Loss 0.9485
Epoch 2 Loss 0.8452
Epoch 3 Loss 0.8217
Epoch 4 Loss 0.8042
Epoch 5 Loss 0.7737
Epoch 6 Loss 0.7673
Epoch 7 Loss 0.7497
Epoch 8 Loss 0.7385
Epoch 9 Loss 0.6970
Epoch 10 Loss 0.6995
Epoch 11 Loss 0.6685
Epoch 12 Loss 0.6354
Epoch 13 Loss 0.6162
Epoch 14 Loss 0.6003
Epoch 15 Loss 0.5496
Epoch 16 Loss 0.5120
Epoch 17 Loss 0.4805
Epoch 18 Loss 0.4725
Epoch 19 Loss 0.4221
Epoch 20 Loss 0.3640
Epoch 21 Loss 0.3437
Epoch 22 Loss 0.3005
Epoch 23 Loss 0.2788
Epoch 24 Loss 0.2358
Epoch 25 Loss 0.2120


In [144]:
from sklearn.metrics import roc_auc_score

model.eval()

with torch.no_grad():

    logits = model(X_test_t, M_test_t, D_test_t, feature_means)

    probs = torch.sigmoid(logits).cpu().numpy()

roc = roc_auc_score(y_test, probs)

print("Test ROC-AUC:", roc)

Test ROC-AUC: 0.7734970852519302


In [145]:
model = GRUD(input_dim=42, hidden_dim=256).to(device)

In [146]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [147]:
epochs = 25

for epoch in range(epochs):

    model.train()
    total_loss = 0

    for xb, mb, db, yb in train_loader:

        optimizer.zero_grad()

        logits = model(xb, mb, db, feature_means)

        loss = criterion(logits, yb)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss {total_loss/len(train_loader):.4f}")

Epoch 1 Loss 0.8994
Epoch 2 Loss 0.8345
Epoch 3 Loss 0.8092
Epoch 4 Loss 0.7859
Epoch 5 Loss 0.7752
Epoch 6 Loss 0.7516
Epoch 7 Loss 0.7280
Epoch 8 Loss 0.7110
Epoch 9 Loss 0.6879
Epoch 10 Loss 0.6655
Epoch 11 Loss 0.6271
Epoch 12 Loss 0.6120
Epoch 13 Loss 0.5772
Epoch 14 Loss 0.5440
Epoch 15 Loss 0.4840
Epoch 16 Loss 0.4427
Epoch 17 Loss 0.3915
Epoch 18 Loss 0.3542
Epoch 19 Loss 0.3135
Epoch 20 Loss 0.2585
Epoch 21 Loss 0.2310
Epoch 22 Loss 0.2115
Epoch 23 Loss 0.2215
Epoch 24 Loss 0.1303
Epoch 25 Loss 0.0790


In [148]:
model.eval()

with torch.no_grad():

    logits = model(X_test_t, M_test_t, D_test_t, feature_means)

    probs = torch.sigmoid(logits).cpu().numpy()

from sklearn.metrics import roc_auc_score
roc = roc_auc_score(y_test, probs)

print("Test ROC-AUC:", roc)

Test ROC-AUC: 0.8144758936875374
